In [3]:
%pip install -r requirements.txt
# Restart the kernel if packages were changed, then continue with the next cell.


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from concurrent.futures import ThreadPoolExecutor

# This notebook uses PyTorch only. Set before importing transformers.
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import cv2
import numpy as np
import torch
import torch.nn as nn
from transformers import (
    AutoModel,
    AutoTokenizer,
    VideoMAEImageProcessor,
    VideoMAEModel,
)

import sentencepiece

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small", use_fast=False)
video_processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")

emotions_to_id = {
    "anger": 0,
    "disgust": 1,
    "fear": 2,
    "joy": 3,
    "neutral": 4,
    "sadness": 5,
    "surprise": 6,
}

cv2.setNumThreads(1)
video_pool = ThreadPoolExecutor(max_workers=4)


def read_video(path, num_frames):
    capture = cv2.VideoCapture(path)
    try:
        if not capture.isOpened():
            raise ValueError(f"Cannot open video: {path}")

        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        if frame_count <= 0:
            raise ValueError(f"Video has no readable frames: {path}")

        target_indices = np.linspace(
            0,
            frame_count - 1,
            num_frames,
        ).astype(int)

        frames = []
        target_position = 0
        for frame_index in range(frame_count):
            ok, frame = capture.read()
            if not ok:
                break

            while (
                target_position < num_frames
                and target_indices[target_position] == frame_index
            ):
                frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                target_position += 1

            if target_position == num_frames:
                break

        if len(frames) != num_frames:
            raise ValueError(
                f"Decoded {len(frames)} of {num_frames} frames from {path}"
            )
        return frames
    finally:
        capture.release()


class MultimodalModel(nn.Module):
    def __init__(self, text_model, visual_model):
        super().__init__()
        self.text_model = text_model
        self.visual_model = visual_model

        # Only train the fusion classifier.
        for param in self.text_model.parameters():
            param.requires_grad = False
        for param in self.visual_model.parameters():
            param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(
                text_model.config.hidden_size + visual_model.config.hidden_size,
                512,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, len(emotions_to_id)),
        )

    def train(self, mode=True):
        super().train(mode)
        # Frozen encoders should remain deterministic; classifier dropout still trains.
        self.text_model.eval()
        self.visual_model.eval()
        return self

    def forward(self, text, visual):
        device = next(self.classifier.parameters()).device
        text_tokens = tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)

        num_frames = self.visual_model.config.num_frames
        frames = list(
            video_pool.map(
                lambda path: read_video(path, num_frames),
                visual,
            )
        )
        visual_tokens = video_processor(frames, return_tensors="pt").to(device)

        with torch.no_grad():
            text_hidden = self.text_model(**text_tokens).last_hidden_state
            mask = text_tokens["attention_mask"].unsqueeze(-1).to(text_hidden.dtype)
            text = (text_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            visual = self.visual_model(
                **visual_tokens
            ).last_hidden_state.mean(dim=1)

        combined = torch.cat([text, visual], dim=-1)
        return self.classifier(combined)


In [2]:
from pathlib import Path

if "MultimodalModel" not in globals():
    raise RuntimeError("Run the tokenizer / video reader / MultimodalModel definition cell first.")

checkpoint_path = Path("meld_classifier_baseline.pt").resolve()


checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

inference_device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

text_model = AutoModel.from_pretrained("microsoft/deberta-v3-small")
visual_model = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base")
inference_model = MultimodalModel(text_model=text_model, visual_model=visual_model)
inference_model.classifier.load_state_dict(checkpoint["classifier"])
inference_model = inference_model.to(inference_device).eval()

import json
import random

with open("responses.json", 'r', encoding='utf-8') as file:
    data = json.load(file)
responses = data["responses"]



import time

@torch.inference_mode()
def predict_emotion(transcript, video_path):
    transcript = transcript.strip()
    if not transcript:
        raise ValueError("Enter the words spoken in the clip.")
    if not str(video_path).strip():
        raise ValueError("Enter a local video file path.")
    path = Path(str(video_path).strip()).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Video not found: {path}")
    # forward() expects batches, even when predicting just one clip.
    inference_model.eval()

    start_time = time.time()
    logits = inference_model([transcript], [str(path)])
    probabilities = logits.softmax(dim=-1)[0].cpu().tolist()

    end_time = time.time()

    list_of_prob = sorted(
        [(emotion, probabilities[label_id]) for emotion, label_id in emotions_to_id.items()],
        key=lambda item: item[1],
        reverse=True,
    )
    random_index = random.randint(0,2)
    emotion = list_of_prob[0][0]
    confidence = list_of_prob[0][1]
    response = responses[emotion][random_index]
    latency = end_time - start_time

    if confidence > 0.4:
        return {
            "emotion": emotion,
            "confidence": confidence,
            "response": response,
            "latency": latency
        }
    else:
        return "No emotion confidence exceeds 0.4", list_of_prob


print(f"Ready on {inference_device}. Relative video paths start at: {Path.cwd()}")


Ready on mps. Relative video paths start at: /Users/macbooka/Documents/VsCode/HuComIntern


In [3]:
import json

import ipywidgets as widgets
from IPython.display import clear_output, display


transcript_input = widgets.Textarea(
    description="Transcript:",
    placeholder="Enter the words spoken in the video...",
    layout=widgets.Layout(width="700px", height="90px"),
)

video_input = widgets.Text(
    description="Video:",
    placeholder="test1.mp4 or /absolute/path/video.mp4",
    layout=widgets.Layout(width="700px"),
)

predict_button = widgets.Button(
    description="Predict emotion",
    button_style="primary",
    icon="play",
)

output = widgets.Output()


def handle_prediction(_):
    predict_button.disabled = True

    with output:
        clear_output(wait=True)

        try:
            result = predict_emotion(
                transcript_input.value,
                video_input.value,
            )
            print(json.dumps(result, indent=2))
        except Exception as error:
            print(f"Prediction failed: {error}")
        finally:
            predict_button.disabled = False


predict_button.on_click(handle_prediction)

display(
    widgets.VBox([
        widgets.HTML("<h3>Multimodal Emotion Demo</h3>"),
        transcript_input,
        video_input,
        predict_button,
        output,
    ])
)